# NB-04 | IAE Threat Scoring & Prioritisation
**Pipeline Step 4 of 5**

Scores each threat using the IAE (Impact + Exploitability + Exposure) model.

## Scoring Formula

| Dimension | Factors | Range |
|-----------|---------|-------|
| **Impact** | Data Sensitivity + Privilege Level + System Criticality | 3–9 |
| **Exploitability** | Access Vector + Input Control + Attack Complexity | 3–9 |
| **Exposure** | Endpoint Visibility + Auth Barrier + Data Flow Reach | 3–9 |

`Total_Raw = Impact + Exploitability + Exposure` (9–27)  
`Final_Score = (Total_Raw − 9) / 18 × 10` (0.0–10.0)

**Severity:** CRITICAL ≥ 23 raw | HIGH ≥ 18 | MEDIUM ≥ 13 | LOW < 13

**Key improvement over v1:**  
Scoring reads the structured `iae_factors` dict that the LLM explicitly output in NB03 — not keyword-matched from threat prose. This makes scores phrasing-independent and reproducible. `pipeline_utils.compute_iae_score` validates and clamps each factor, logging any out-of-range values.

**Input:** `threats.json`, `repo_surface.json`  
**Output:** `scored_threats.json`

In [ ]:
import sys
sys.path.insert(0, ".")

from pipeline_utils import (
    load_json, save_json, get_logger,
    compute_iae_score, filter_grounded_threats,
)

from collections import Counter
from pathlib import Path

log = get_logger("NB04")

## 4.1 — Load Inputs

In [ ]:
surface      = load_json("repo_surface.json",  schema_keys=["valid_paths", "run_id"])
threats_data = load_json("threats.json",        schema_keys=["stride_results", "metadata"])

# Run ID consistency check
if threats_data["metadata"]["run_id"] != surface["run_id"]:
    log.warning(
        "run_id mismatch: surface=%s threats=%s — possible stale artifact",
        surface["run_id"], threats_data["metadata"]["run_id"]
    )

valid_paths: set[str] = set(surface["valid_paths"])
RUN_ID = surface["run_id"]

log.info("Loaded threats from run: %s", threats_data["metadata"]["run_id"])

## 4.2 — Flatten & Secondary Grounding Pass

NB03 already filtered threats, but we apply a final pass here in case any slipped through (e.g. if NB03 was run with a different `valid_paths` set).

In [ ]:
# Flatten all component threat lists into a single list with component metadata
flat: list[dict] = []
for comp_id, data in threats_data["stride_results"].items():
    for threat in data["threats"]:
        # Copy threat so we don't mutate the original nested structure
        t = dict(threat)
        t["component_id"]   = comp_id
        t["component_name"] = data["component_name"]
        t["component_type"] = data.get("component_type", "service")
        flat.append(t)

log.info("Total threats to score: %d", len(flat))

# Secondary grounding pass
flat = filter_grounded_threats(flat, valid_paths, label="secondary-pass", logger=log)
log.info("Threats after secondary filter: %d", len(flat))

## 4.3 — Compute IAE Scores

`compute_iae_score` reads `threat["iae_factors"]` directly — no prose parsing. If a threat is missing `iae_factors`, validation-clamped defaults (all 2s) are used and a warning is logged.

In [ ]:
missing_factors_count = 0

for threat in flat:
    if not threat.get("iae_factors"):
        missing_factors_count += 1
        log.warning(
            "Threat '%s' (%s) has no iae_factors — defaults will be used",
            threat.get("title", "?"), threat.get("component_name", "?")
        )
        # Provide a neutral default so scoring doesn't crash
        threat["iae_factors"] = {
            "data_sensitivity":      2,
            "privilege_level":       2,
            "system_criticality":    2,
            "access_vector":         2,
            "exploit_input_control": 2,
            "attack_complexity":     2,
            "endpoint_visibility":   2,
            "auth_barrier":          2,
            "data_flow_reach":       2,
        }

    scores = compute_iae_score(threat)
    threat.update(scores)   # merges final_score, severity, sub-scores, factor_breakdown

if missing_factors_count:
    log.warning(
        "%d threats lacked iae_factors and were scored with defaults. "
        "Re-run NB03 to get accurate scores.",
        missing_factors_count
    )

flat.sort(key=lambda x: -x["total_raw"])

dist = Counter(t["severity"] for t in flat)
log.info("CRITICAL: %d  HIGH: %d  MEDIUM: %d  LOW: %d  Total: %d",
         dist["CRITICAL"], dist["HIGH"], dist["MEDIUM"], dist["LOW"], len(flat))

## 4.4 — Top Threats with Factor Breakdown

In [ ]:
SEV_LABELS = {"CRITICAL": "!! CRITICAL", "HIGH": "HIGH    ", "MEDIUM": "MEDIUM  ", "LOW": "LOW     "}

print(f"{'SCR':>4}  {'SEVERITY':12}  {'COMPONENT':22}  THREAT")
print("-" * 85)
for t in flat[:20]:
    sev = SEV_LABELS.get(t["severity"], t["severity"])
    print(f"{t['final_score']:4.1f}  {sev:12}  {t['component_name'][:22]:22}  {t['title']}")

if flat:
    print("\n── Factor Breakdown for top threat ──")
    top = flat[0]
    fb  = top["factor_breakdown"]
    print(f"  Threat  : {top['title']}")
    print(f"  Location: {top.get('threat_location', 'N/A')}")
    print(f"  STRIDE  : {top.get('stride_category', '?')}")
    print(f"  Source  : structured iae_factors (not keyword-inferred)")
    print()
    for dim, factors in [
        ("IMPACT",          [("data_sensitivity",    "Data Sensitivity"),
                             ("privilege_level",     "Privilege Level"),
                             ("system_criticality",  "System Criticality")]),
        ("EXPLOITABILITY",  [("access_vector",       "Access Vector"),
                             ("exploit_input_control","Input Control"),
                             ("attack_complexity",   "Attack Complexity")]),
        ("EXPOSURE",        [("endpoint_visibility", "Endpoint Visibility"),
                             ("auth_barrier",         "Auth Barrier"),
                             ("data_flow_reach",      "Data Flow Reach")]),
    ]:
        sub = {"IMPACT": top["impact_score"], "EXPLOITABILITY": top["exploit_score"],
               "EXPOSURE": top["exposure_score"]}[dim]
        print(f"  {dim} ({sub}/9)")
        for key, label in factors:
            print(f"    {label:22}: {fb.get(key, '?')} / 3")
    print(f"  {'─'*40}")
    print(f"  Total Raw  : {top['total_raw']} / 27")
    print(f"  Final Score: {top['final_score']} / 10")
    print(f"  Severity   : {top['severity']}")

## 4.5 — Save `scored_threats.json`

In [ ]:
output = {
    "threats": flat,
    "summary": {s: dist[s] for s in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]},
    "metadata": {
        "run_id":                  RUN_ID,
        "repo":                    surface["repo"],
        "total_scored":            len(flat),
        "missing_iae_factors":     missing_factors_count,
        "scoring_model":           "IAE — structured iae_factors (not keyword-inferred)",
        "scoring_source":          "pipeline_utils.compute_iae_score",
    },
}

save_json(output, "scored_threats.json", run_id=RUN_ID)
print("Saved scored_threats.json")